In [ ]:

import pandas as pd

In [ ]:
import sys
print(sys.executable)

In [ ]:
df = pd.read_excel("opensooq_cars.xlsx")

In [ ]:
df.columns

In [ ]:
df = df.sort_values(
    by=["Brand", "Model", "Year"],
    ascending=[True, True, False]
)

In [ ]:
df

In [ ]:
df["Body Type"].isnull().sum()

In [ ]:
# Step 1: Filter rows that are valid for computing the average
# - Price > 0 (drop the fake zeros)
# - Payment Method == 'Cash' (only real prices)
valid = df[(df['Price'] > 0) & (df['Payment Method'] == 'Cash')]

# Step 2: Compute average cash price per car group
group_cols = ['Brand', 'Model', 'Year', 'Trim']
avg_prices = valid.groupby(group_cols, dropna=False)['Price'].mean().reset_index()
avg_prices = avg_prices.rename(columns={'Price': 'final_price'})

# Step 3: Merge back to the original df so every row gets its group's avg
df = df.merge(avg_prices, on=group_cols, how='left')

In [ ]:
# How many rows ended up with no final_price?
print("Rows missing final_price:", df['final_price'].isna().sum())

# Peek at the result
df[['Brand', 'Model', 'Year', 'Trim', 'Price', 'Payment Method', 'final_price']].head(10)

In [ ]:
# Drop final_price if it already exists from a previous run
df = df.drop(columns=['final_price'], errors='ignore')

# Then run the original code
valid = df[(df['Price'] > 0) & (df['Payment Method'] == 'Cash')]

group_cols = ['Brand', 'Model', 'Year', 'Trim']
avg_prices = (
    valid.groupby(group_cols, dropna=False)['Price']
         .mean()
         .reset_index()
         .rename(columns={'Price': 'final_price'})
)

df = df.merge(avg_prices, on=group_cols, how='left')
df = df.dropna(subset=['final_price']).reset_index(drop=True)

print(f"Rows remaining: {len(df)}")

In [ ]:
# Make sure we're working with valid cash listings (same filter as before)
valid = df[(df['Price'] > 0) & (df['Payment Method'] == 'Cash')].copy()

group_cols = ['Brand', 'Model', 'Year', 'Trim']

# Compute group stats
stats = valid.groupby(group_cols, dropna=False)['Price'].agg(['mean', 'median', 'std', 'count']).reset_index()
valid = valid.merge(stats, on=group_cols, how='left')

# Method 1: Z-score (how many std devs away from group mean)
valid['z_score'] = (valid['Price'] - valid['mean']) / valid['std']

# Method 2: Ratio to median (more robust against extreme outliers skewing the mean)
valid['price_ratio'] = valid['Price'] / valid['median']

# Flag outliers: price less than 30% of group median OR more than 3x
valid['is_outlier'] = (valid['price_ratio'] < 0.3) | (valid['price_ratio'] > 3)

# View outliers, sorted by how extreme
outliers = valid[valid['is_outlier']].sort_values('price_ratio')
outliers[['Brand', 'Model', 'Year', 'Trim', 'Price', 'median', 'mean', 'count', 'price_ratio', 'Link']]

In [ ]:
# Drop final_price if it exists from a previous run (idempotent)
df = df.drop(columns=['final_price'], errors='ignore')

group_cols = ['Brand', 'Model', 'Year', 'Trim']

# Step 1: Start from valid rows (Cash + nonzero price) for computing the reference
valid = df[(df['Price'] > 0) & (df['Payment Method'] == 'Cash')].copy()

# Step 2: Compute group median to detect outliers
valid['group_median'] = valid.groupby(group_cols, dropna=False)['Price'].transform('median')
valid['price_ratio'] = valid['Price'] / valid['group_median']

# Step 3: Keep only non-outlier rows for the average calculation
clean = valid[(valid['price_ratio'] >= 0.3) & (valid['price_ratio'] <= 3)]

# Step 4: Compute clean average per group
avg_prices = (
    clean.groupby(group_cols, dropna=False)['Price']
         .mean()
         .reset_index()
         .rename(columns={'Price': 'final_price'})
)

# Step 5: Merge back — every row in df gets its group's clean average
df = df.merge(avg_prices, on=group_cols, how='left')

# Step 6: Drop rows where final_price couldn't be computed (group had no valid cash listings after filtering)
df = df.dropna(subset=['final_price']).reset_index(drop=True)

print(f"Rows remaining: {len(df)}")
print(f"Unique car groups: {df.groupby(group_cols, dropna=False).ngroups}")

In [ ]:
# Verify no extreme final_price values remain
print(df['final_price'].describe())

# Spot-check a few groups to confirm
df[df['Brand'] == 'Mercedes Benz'][['Brand', 'Model', 'Year', 'Trim', 'Price', 'Payment Method', 'final_price']].head(10)

In [ ]:
df['engine_size_cc'] = (
    df['Engine Size']
    .astype(str)
    .str.replace(',', '', regex=False)   # remove thousand separators
    .str.extract(r'(\d+)\s*cc')          # grab the number right before "cc"
    .astype(float)
)

In [ ]:
df

In [ ]:
group_cols = ['Brand', 'Model', 'Year', 'Trim']

# Count rows per group
group_sizes = df.groupby(group_cols, dropna=False).size().reset_index(name='count')

# Keep only groups that appeared exactly twice
twice_groups = group_sizes[group_sizes['count'] == 2][group_cols]

# Filter the original df to those groups
result = df.merge(twice_groups, on=group_cols, how='inner')

# Sort so duplicates appear together
result = result.sort_values(group_cols).reset_index(drop=True)

print(f"Found {len(result)} rows ({len(twice_groups)} car groups appearing twice)")
result

In [ ]:
df.to_excel('cars_cleaned.xlsx', index=False)

In [ ]:
# Copy 1: Full dataset for Power BI (keep everything, including NaN Engine Size rows)
df_powerbi = df.copy()

# Copy 2: Model training set (drop rows where engine_size_cc is NaN)
df_model = df.dropna(subset=['Transmission']).reset_index(drop=True)

print(f"Power BI dataset: {len(df_powerbi)} rows")
print(f"Model training dataset: {len(df_model)} rows")
print(f"Rows dropped for model: {len(df_powerbi) - len(df_model)}")

In [ ]:
df_powerbi.columns

In [ ]:
# ============================================================
# Fix dtypes on both dfs (Year, Mileage, Seats came in as strings)
# ============================================================

# --- Power BI df ---
df_powerbi['Year'] = pd.to_numeric(df_powerbi['Year'], errors='coerce')
df_powerbi['Mileage'] = pd.to_numeric(df_powerbi['Mileage'], errors='coerce')
df_powerbi['Seats'] = pd.to_numeric(df_powerbi['Seats'], errors='coerce')

# --- Model df ---
df_model['Year'] = pd.to_numeric(df_model['Year'], errors='coerce')
df_model['Mileage'] = pd.to_numeric(df_model['Mileage'], errors='coerce')
df_model['Seats'] = pd.to_numeric(df_model['Seats'], errors='coerce')

# ============================================================
# Apply the Year >= 1970 filter (model df only)
# ============================================================
df_model = df_model[df_model['Year'] >= 1970].reset_index(drop=True)

# ============================================================
# Sanity check
# ============================================================
print(f"Power BI dataset: {len(df_powerbi)} rows, {df_powerbi.shape[1]} columns")
print(f"Model dataset:    {len(df_model)} rows, {df_model.shape[1]} columns")

print("\nPower BI dtypes:")
print(df_powerbi.dtypes)

print("\nModel dtypes:")
print(df_model.dtypes)

In [ ]:
import re

def parse_mileage(val):
    """Convert OpenSooq mileage strings to integer (upper bound)."""
    if pd.isna(val):
        return pd.NA
    
    s = str(val).replace(',', '').strip()
    
    # Extract all numbers from the string
    nums = re.findall(r'\d+', s)
    if not nums:
        return pd.NA
    
    nums = [int(n) for n in nums]
    
    # Take the largest number (handles "1-999" → 999, "+200000" → 200000, "50000" → 50000)
    return max(nums)

# Apply to both dfs
df_powerbi['Mileage'] = df_powerbi['Mileage'].apply(parse_mileage).astype('Int64')
df_model['Mileage']   = df_model['Mileage'].apply(parse_mileage).astype('Int64')

# Quick check
print("Power BI Mileage sample:")
print(df_powerbi['Mileage'].describe())
print("\nModel Mileage sample:")
print(df_model['Mileage'].describe())

In [ ]:
group_cols = ['Brand', 'Model', 'Year', 'Trim']

def find_mileage_outliers(df, label):
    valid = df[df['Mileage'].notna() & (df['Mileage'] > 0)].copy()
    
    valid['mileage_median'] = valid.groupby(group_cols, dropna=False)['Mileage'].transform('median')
    valid['mileage_count']  = valid.groupby(group_cols, dropna=False)['Mileage'].transform('size')
    valid['mileage_ratio']  = valid['Mileage'] / valid['mileage_median']
    
    # Flag: ratio < 0.3 or > 3, and group has at least 3 listings (so median is trustable)
    valid['is_mileage_outlier'] = (
        ((valid['mileage_ratio'] < 0.3) | (valid['mileage_ratio'] > 3))
        & (valid['mileage_count'] >= 3)
    )
    
    outliers = valid[valid['is_mileage_outlier']].sort_values('mileage_ratio')
    
    print(f"\n=== {label} ===")
    print(f"Total rows: {len(df)}")
    print(f"Outlier rows: {len(outliers)}")
    
    return outliers[['Brand', 'Model', 'Year', 'Trim', 'Mileage',
                     'mileage_median', 'mileage_count', 'mileage_ratio', 'Link']]

outliers_powerbi = find_mileage_outliers(df_powerbi, "Power BI df")
outliers_model   = find_mileage_outliers(df_model,   "Model df")

# Display them
print("\nPower BI outliers:")
print(outliers_powerbi.head(30))

print("\nModel outliers:")
print(outliers_model.head(30))

In [ ]:
group_cols = ['Brand', 'Model', 'Year', 'Trim']

# Compute group median and ratio for mileage
df_model['mileage_median'] = df_model.groupby(group_cols, dropna=False)['Mileage'].transform('median')
df_model['mileage_count']  = df_model.groupby(group_cols, dropna=False)['Mileage'].transform('size')
df_model['mileage_ratio']  = df_model['Mileage'] / df_model['mileage_median']

# Flag outliers (only when group is big enough to trust the median)
is_outlier = (
    ((df_model['mileage_ratio'] < 0.3) | (df_model['mileage_ratio'] > 3))
    & (df_model['mileage_count'] >= 3)
)

print(f"Rows before: {len(df_model)}")
print(f"Outliers to drop: {is_outlier.sum()}")

# Drop outliers
df_model = df_model[~is_outlier].reset_index(drop=True)

# Clean up the helper columns
df_model = df_model.drop(columns=['mileage_median', 'mileage_count', 'mileage_ratio'])

print(f"Rows after:  {len(df_model)}")

In [ ]:
# Identify all string (object) columns to clean
def clean_text_columns(df):
    """Strip whitespace and normalize casing on all string columns."""
    text_cols = df.select_dtypes(include='object').columns
    
    for col in text_cols:
        # Skip URL columns — casing matters for links
        if col in ['Link', 'Image']:
            continue
        
        df[col] = (
            df[col]
            .astype(str)                          # safety for mixed types
            .str.strip()                          # trim leading/trailing whitespace
            .replace({'nan': pd.NA, '': pd.NA})   # restore real NaNs
        )
    
    return df

# Apply to both
df_powerbi = clean_text_columns(df_powerbi)
df_model   = clean_text_columns(df_model)

# Now apply Title Case to specific columns where it makes sense
title_case_cols = [
    'Brand', 'Model', 'Trim', 'Body Type', 'Condition',
    'Transmission', 'Fuel', 'Exterior Color', 'Interior Color',
    'Body Condition', 'Paint', 'Payment Method',
    'Neighbourhood', 'City'
]

def apply_title_case(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].str.title()
    return df

df_powerbi = apply_title_case(df_powerbi, title_case_cols)
df_model   = apply_title_case(df_model, title_case_cols)

# Quick check — see if any column collapsed nicely
print("Body Type values (model df):")
print(df_model['Body Type'].value_counts(dropna=False))

print("\nFuel values (model df):")
print(df_model['Fuel'].value_counts(dropna=False))

print("\nTransmission values (model df):")
print(df_model['Transmission'].value_counts(dropna=False))

In [ ]:
df_model = df_model.drop(columns=['Payment Method', 'Neighbourhood', 'Link'], errors='ignore')

print(f"Model columns now: {list(df_model.columns)}")

In [ ]:
df_model

In [ ]:
# Drop columns not useful for price prediction


# Reorder columns: identifiers → vehicle specs → condition/appearance → location → target
column_order = [
    # Identifiers / categorical core
    'Brand',
    'Model',
    'Trim',
    'Year',
    'Body Type',
    
    # Vehicle specs (numeric + categorical)
    'Mileage',
    'engine_size_cc',
    'Fuel',
    
    # Condition & appearance
    'Condition',
    'Body Condition',
    'Paint',
    'Exterior Color',
    'Interior Color',
    
    # Location
    'City',
    
    # Targets (keep both, you'll pick one)
    'Price',
    'final_price',
]

df_model = df_model[column_order]

print(df_model.shape)
df_model.head()

In [ ]:
df_model = df_model.drop(columns=[ 'Price'])

In [ ]:
df_model

In [ ]:
cols_to_fill = ['Paint', 'Exterior Color', 'Interior Color']

def fill_with_group_mode(df, col, group_cols):
    """Fill NaN in `col` using the mode within `group_cols`."""
    return df.groupby(group_cols)[col].transform(
        lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
    )

# Try progressively broader groupings as fallback
for col in cols_to_fill:
    print(f"\n--- {col} ---")
    print(f"Nulls before: {df_model[col].isnull().sum()}")

    # 1st pass: Brand + Model + Year
    df_model[col] = fill_with_group_mode(df_model, col, ['Brand', 'Model', 'Year'])
    print(f"After Brand+Model+Year: {df_model[col].isnull().sum()}")

    # 2nd pass: Brand + Model (broader)
    df_model[col] = fill_with_group_mode(df_model, col, ['Brand', 'Model'])
    print(f"After Brand+Model: {df_model[col].isnull().sum()}")

    # 3rd pass: Brand only (broadest)
    df_model[col] = fill_with_group_mode(df_model, col, ['Brand'])
    print(f"After Brand: {df_model[col].isnull().sum()}")

    # 4th pass: global mode (last resort)
    if df_model[col].isnull().any():
        global_mode = df_model[col].mode().iloc[0]
        df_model[col] = df_model[col].fillna(global_mode)
        print(f"After global mode ('{global_mode}'): {df_model[col].isnull().sum()}")

In [ ]:
import numpy as np

# Convert to float64 to allow NaN and median values
df_model['Mileage'] = df_model['Mileage'].astype('float64')

print("=== Mileage ===")
print(f"Nulls before: {df_model['Mileage'].isnull().sum()}")

# Rule A: Year >= 2025 → assume new
mask_new = (df_model['Year'] >= 2025) & (df_model['Mileage'].isnull())
df_model.loc[mask_new, 'Mileage'] = 0.0
print(f"Filled as new (Year>=2025): {mask_new.sum()}")

# Rule B: Condition == 'New' → mileage = 0
mask_cond_new = (df_model['Condition'] == 'New') & (df_model['Mileage'].isnull())
df_model.loc[mask_cond_new, 'Mileage'] = 0.0
print(f"Filled as new (Condition=New): {mask_cond_new.sum()}")

# Rule C: median by Brand+Model+Year, then fallbacks
def fill_with_group_median(df, col, group_cols):
    return df.groupby(group_cols)[col].transform(lambda x: x.fillna(x.median()))

df_model['Mileage'] = fill_with_group_median(df_model, 'Mileage', ['Brand', 'Model', 'Year'])
print(f"After Brand+Model+Year median: {df_model['Mileage'].isnull().sum()}")

df_model['Mileage'] = fill_with_group_median(df_model, 'Mileage', ['Brand', 'Year'])
print(f"After Brand+Year median: {df_model['Mileage'].isnull().sum()}")

df_model['Mileage'] = fill_with_group_median(df_model, 'Mileage', ['Year'])
print(f"After Year median: {df_model['Mileage'].isnull().sum()}")

if df_model['Mileage'].isnull().any():
    df_model['Mileage'] = df_model['Mileage'].fillna(df_model['Mileage'].median())
    print(f"After global median: {df_model['Mileage'].isnull().sum()}")

# Optional: convert back to int after all NaNs are filled
df_model['Mileage'] = df_model['Mileage'].round().astype('int64')
print(f"Final dtype: {df_model['Mileage'].dtype}")

In [ ]:
print("=== Body Condition ===")
print(f"Nulls before: {df_model['Body Condition'].isnull().sum()}")

# Rule A: Mileage == 0 → treat as new, body condition is pristine
mask_zero_mileage = (df_model['Mileage'] == 0) & (df_model['Body Condition'].isnull())
df_model.loc[mask_zero_mileage, 'Body Condition'] = 'Excellent With No Defects'
print(f"Filled as new (Mileage=0): {mask_zero_mileage.sum()}")

# Rule B: Mileage > 0 → used cars, fill with mode by Brand+Model+Year
def fill_with_group_mode(df, col, group_cols):
    return df.groupby(group_cols)[col].transform(
        lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
    )

df_model['Body Condition'] = fill_with_group_mode(df_model, 'Body Condition', ['Brand', 'Model', 'Year'])
print(f"After Brand+Model+Year mode: {df_model['Body Condition'].isnull().sum()}")

# Fallback: Year only
df_model['Body Condition'] = fill_with_group_mode(df_model, 'Body Condition', ['Year'])
print(f"After Year mode: {df_model['Body Condition'].isnull().sum()}")

# Last resort: global mode
if df_model['Body Condition'].isnull().any():
    global_mode = df_model['Body Condition'].mode().iloc[0]
    df_model['Body Condition'] = df_model['Body Condition'].fillna(global_mode)
    print(f"After global mode ('{global_mode}'): {df_model['Body Condition'].isnull().sum()}")

print(f"\nFinal nulls: {df_model['Body Condition'].isnull().sum()}")

In [ ]:
print("=== Trim ===")
print(f"Nulls before: {df_model['Trim'].isnull().sum()}")

# Fill all NaN trims with 'Base' — treats missing trim as its own category
df_model['Trim'] = df_model['Trim'].fillna('Base')

print(f"Nulls after: {df_model['Trim'].isnull().sum()}")
print(f"\nTop 10 trims:")
print(df_model['Trim'].value_counts().head(10))

In [ ]:
print("=== engine_size_cc ===")
print(f"Nulls before: {df_model['engine_size_cc'].isnull().sum()}")

# Make sure dtype is float to avoid the Int64 issue from before
df_model['engine_size_cc'] = df_model['engine_size_cc'].astype('float64')

# Rule A: Electric → 0 (no combustion engine)
mask_electric = (df_model['Fuel'] == 'Electric') & (df_model['engine_size_cc'].isnull())
df_model.loc[mask_electric, 'engine_size_cc'] = 0.0
print(f"Filled as Electric (engine=0): {mask_electric.sum()}")

# Rule B: Hybrid cars often have engines too, but some listings omit it
# Treat them like ICE cars — use median imputation. (Only set to 0 if you're sure they're plug-in EVs.)

# Rule C: Non-Electric → median by Brand+Model+Year
def fill_with_group_median(df, col, group_cols):
    return df.groupby(group_cols)[col].transform(lambda x: x.fillna(x.median()))

df_model['engine_size_cc'] = fill_with_group_median(df_model, 'engine_size_cc', ['Brand', 'Model', 'Year'])
print(f"After Brand+Model+Year median: {df_model['engine_size_cc'].isnull().sum()}")

# Fallback: Brand + Model (different year, same model usually has same engine family)
df_model['engine_size_cc'] = fill_with_group_median(df_model, 'engine_size_cc', ['Brand', 'Model'])
print(f"After Brand+Model median: {df_model['engine_size_cc'].isnull().sum()}")

# Fallback: Brand + Body Type (e.g., Toyota SUV → typical engine size)
df_model['engine_size_cc'] = fill_with_group_median(df_model, 'engine_size_cc', ['Brand', 'Body Type'])
print(f"After Brand+Body Type median: {df_model['engine_size_cc'].isnull().sum()}")

# Fallback: Body Type only (SUVs tend to have bigger engines than sedans)
df_model['engine_size_cc'] = fill_with_group_median(df_model, 'engine_size_cc', ['Body Type'])
print(f"After Body Type median: {df_model['engine_size_cc'].isnull().sum()}")

# Last resort: global median
if df_model['engine_size_cc'].isnull().any():
    df_model['engine_size_cc'] = df_model['engine_size_cc'].fillna(df_model['engine_size_cc'].median())
    print(f"After global median: {df_model['engine_size_cc'].isnull().sum()}")

print(f"\nFinal nulls: {df_model['engine_size_cc'].isnull().sum()}")
print(f"Distribution:\n{df_model['engine_size_cc'].describe()}")

In [ ]:
# 1. Quick count of nulls per column
df_model.isnull().sum().sort_values(ascending=False)

In [ ]:
df_model.columns = df_model.columns.str.lower().str.replace(' ', '_')

print(df_model.columns.tolist())

In [ ]:
# For Power BI — Excel keeps Arabic text clean and is easy to import
df_powerbi.to_excel('cars_powerbi.xlsx', index=False)

# For model training — CSV is lighter and faster for KNIME / pandas reload
df_model.to_csv('cars_model.csv', index=False, encoding='utf-8-sig')